In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Define altcoins and parameters
altcoins = ['ETH-USD', 'BNB-USD', 'ADA-USD', 'XRP-USD', 'SOL-USD']  # Example altcoins
start_date = (datetime.now() - timedelta(days=45)).strftime('%Y-%m-%d')  # 2 years of data
end_date = datetime.now().strftime('%Y-%m-%d')
short_ma = 50  # Short-term MA period
long_ma = 200   # Long-term MA period
initial_capital = 10000  # Starting capital in USD

# Download data
data = yf.download(altcoins, start=start_date, end=end_date, interval = '1h', group_by='ticker')

# Create equally-weighted index
def create_index(data, altcoins):
    close_prices = pd.DataFrame()
    for coin in altcoins:
        close_prices[coin] = data[coin]['Close']
    # Calculate returns and create index
    returns = close_prices.pct_change().mean(axis=1)  # Equal-weighted average returns
    index = (1 + returns).cumprod() * 100  # Normalize to 100
    index = index.fillna(method='ffill')  # Forward fill any NaN
    return index, close_prices

index, close_prices = create_index(data, altcoins)

# Calculate moving averages
index_df = pd.DataFrame(index, columns=['Index'])
index_df['Short_MA'] = index_df['Index'].rolling(window=short_ma).mean()
index_df['Long_MA'] = index_df['Index'].rolling(window=long_ma).mean()

# Generate signals
index_df['Signal'] = 0
index_df['Signal'][short_ma:] = np.where(
    index_df['Short_MA'][short_ma:] > index_df['Long_MA'][short_ma:], 1, 0
)
index_df['Position'] = index_df['Signal'].diff()

# Calculate max drawdown
def calculate_max_drawdown(portfolio_value):
    rolling_max = portfolio_value.cummax()
    drawdowns = (portfolio_value - rolling_max) / rolling_max
    max_drawdown = drawdowns.min() * 100  # Convert to percentage
    return max_drawdown

# Backtest strategy
def backtest_strategy(index_df, initial_capital):
    positions = pd.DataFrame(index=index_df.index)
    positions['Position'] = index_df['Signal']

    # Calculate daily returns
    daily_returns = index_df['Index'].pct_change()

    # Calculate strategy returns
    strategy_returns = daily_returns * positions['Position'].shift(1)
    strategy_returns = strategy_returns.fillna(0)

    # Calculate cumulative returns
    portfolio_value = initial_capital * (1 + strategy_returns).cumprod()

    # Calculate performance metrics
    total_return = (portfolio_value[-1] / initial_capital - 1) * 100
    annualized_return = ((portfolio_value[-1] / initial_capital) ** (252 / len(portfolio_value)) - 1) * 100
    sharpe_ratio = (strategy_returns.mean() * 252) / (strategy_returns.std() * np.sqrt(252))
    max_drawdown = calculate_max_drawdown(portfolio_value)

    return portfolio_value, total_return, annualized_return, sharpe_ratio, max_drawdown

# Run backtest
portfolio_value, total_return, annualized_return, sharpe_ratio, max_drawdown = backtest_strategy(index_df, initial_capital)

# Print results
print(f"Backtest Results for Altcoin Index MA Crossover Strategy")
print(f"Period: {start_date} to {end_date}")
print(f"Initial Capital: ${initial_capital:,.2f}")
print(f"Final Portfolio Value: ${portfolio_value[-1]:,.2f}")
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Maximum Drawdown: {max_drawdown:.2f}%")

# Optional: Save results to CSV
portfolio_value.to_csv('portfolio_value.csv')

/tmp/ipython-input-3-1642995920.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(altcoins, start=start_date, end=end_date, interval = '1h', group_by='ticker')
[*********************100%***********************]  5 of 5 completed


Backtest Results for Altcoin Index MA Crossover Strategy
Period: 2025-06-09 to 2025-07-24
Initial Capital: $10,000.00
Final Portfolio Value: $13,642.01
Total Return: 36.42%
Annualized Return: 7.52%
Sharpe Ratio: 1.03
Maximum Drawdown: -7.93%


/tmp/ipython-input-3-1642995920.py:25: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  index = index.fillna(method='ffill')  # Forward fill any NaN
/tmp/ipython-input-3-1642995920.py:37: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org

In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Define altcoins and parameters
altcoins = ['ETH-USD', 'BNB-USD', 'ADA-USD', 'XRP-USD', 'SOL-USD']  # Example altcoins
btc_ticker = 'BTC-USD'
start_date = (datetime.now() - timedelta(days=360)).strftime('%Y-%m-%d')  # 2 years of data
end_date = datetime.now().strftime('%Y-%m-%d')
initial_capital = 10000  # Starting capital in USD
lookback = 20  # Lookback period for z-score calculation
entry_z = 2.0  # Z-score threshold for entering trades
exit_z = 0.5   # Z-score threshold for exiting trades

# Download data
data = yf.download(altcoins + [btc_ticker], start=start_date, end=end_date, interval = '4h',group_by='ticker')

# Create equally-weighted altcoin index
def create_index(data, altcoins):
    close_prices = pd.DataFrame()
    for coin in altcoins:
        close_prices[coin] = data[coin]['Close']
    # Calculate returns and create index
    returns = close_prices.pct_change().mean(axis=1)  # Equal-weighted average returns
    index = (1 + returns).cumprod() * 100  # Normalize to 100
    index = index.fillna(method='ffill')  # Forward fill any NaN
    return index, close_prices

index, close_prices = create_index(data, altcoins)
btc_prices = data[btc_ticker]['Close']

# Calculate spread and z-score
def calculate_spread_and_zscore(index, btc_prices, lookback):
    spread = index - btc_prices  # Spread between index and BTC
    spread_df = pd.DataFrame({'Spread': spread}, index=index.index)
    spread_df['Mean'] = spread_df['Spread'].rolling(window=lookback).mean()
    spread_df['Std'] = spread_df['Spread'].rolling(window=lookback).std()
    spread_df['Z_Score'] = (spread_df['Spread'] - spread_df['Mean']) / spread_df['Std']
    return spread_df

spread_df = calculate_spread_and_zscore(index, btc_prices, lookback)

# Generate trading signals
def generate_signals(spread_df, entry_z, exit_z):
    signals = pd.DataFrame(index=spread_df.index)
    signals['Z_Score'] = spread_df['Z_Score']
    signals['Signal'] = 0

    position = 0  # 1 for long index/short BTC, -1 for short index/long BTC, 0 for no position
    for i in range(1, len(signals)):
        z_score = signals['Z_Score'].iloc[i]
        if position == 0:
            if z_score > entry_z:  # Spread too high, short index, long BTC
                signals.iloc[i, signals.columns.get_loc('Signal')] = -1
                position = -1
            elif z_score < -entry_z:  # Spread too low, long index, short BTC
                signals.iloc[i, signals.columns.get_loc('Signal')] = 1
                position = 1
        elif position == 1 and abs(z_score) < exit_z:  # Exit long position
            signals.iloc[i, signals.columns.get_loc('Signal')] = 0
            position = 0
        elif position == -1 and abs(z_score) < exit_z:  # Exit short position
            signals.iloc[i, signals.columns.get_loc('Signal')] = 0
            position = 0
        else:
            signals.iloc[i, signals.columns.get_loc('Signal')] = position
    return signals

signals = generate_signals(spread_df, entry_z, exit_z)

# Calculate max drawdown
def calculate_max_drawdown(portfolio_value):
    rolling_max = portfolio_value.cummax()
    drawdowns = (portfolio_value - rolling_max) / rolling_max
    max_drawdown = drawdowns.min() * 100  # Convert to percentage
    return max_drawdown

# Backtest strategy
def backtest_strategy(signals, index, btc_prices, initial_capital):
    portfolio = pd.DataFrame(index=signals.index)
    portfolio['Signal'] = signals['Signal']

    # Calculate returns for index and BTC
    index_returns = index.pct_change()
    btc_returns = btc_prices.pct_change()

    # Strategy returns: long index/short BTC or short index/long BTC
    strategy_returns = portfolio['Signal'].shift(1) * (index_returns - btc_returns)
    strategy_returns = strategy_returns.fillna(0)

    # Calculate cumulative returns
    portfolio_value = initial_capital * (1 + strategy_returns).cumprod()

    # Calculate performance metrics
    total_return = (portfolio_value[-1] / initial_capital - 1) * 100
    annualized_return = ((portfolio_value[-1] / initial_capital) ** (252 / len(portfolio_value)) - 1) * 100
    sharpe_ratio = (strategy_returns.mean() * 252) / (strategy_returns.std() * np.sqrt(252)) if strategy_returns.std() != 0 else 0
    max_drawdown = calculate_max_drawdown(portfolio_value)

    return portfolio_value, total_return, annualized_return, sharpe_ratio, max_drawdown

# Run backtest
portfolio_value, total_return, annualized_return, sharpe_ratio, max_drawdown = backtest_strategy(signals, index, btc_prices, initial_capital)

# Print results
print(f"Backtest Results for Statistical Arbitrage: Altcoin Index vs BTC-USD")
print(f"Period: {start_date} to {end_date}")
print(f"Initial Capital: ${initial_capital:,.2f}")
print(f"Final Portfolio Value: ${portfolio_value[-1]:,.2f}")
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Maximum Drawdown: {max_drawdown:.2f}%")

# Optional: Save results to CSV
portfolio_value.to_csv('stat_arb_portfolio_value.csv')

/tmp/ipython-input-10-2050449422.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(altcoins + [btc_ticker], start=start_date, end=end_date, interval = '4h',group_by='ticker')
[*********************100%***********************]  6 of 6 completed
/tmp/ipython-input-10-2050449422.py:27: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  index = index.fillna(method='ffill')  # Forward fill any NaN
/tmp/ipython-input-10-2050449422.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_return = (portfolio_value[-1] / initial_capital - 1) * 100
/tmp/ipython-input-10-2050449422.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a futu

Backtest Results for Statistical Arbitrage: Altcoin Index vs BTC-USD
Period: 2024-07-29 to 2025-07-24
Initial Capital: $10,000.00
Final Portfolio Value: $14,533.91
Total Return: 45.34%
Annualized Return: 4.46%
Sharpe Ratio: 0.45
Maximum Drawdown: -15.24%


In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt

# --- Black-Scholes & Implied Volatility ---

def bs_price(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)
    else:
        return K * np.exp(-r*T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def implied_volatility(market_price, S, K, T, r=0, option_type='call'):
    try:
        return brentq(lambda sigma: bs_price(S, K, T, r, sigma, option_type) - market_price,
                      1e-6, 3.0, maxiter=1000)
    except:
        return np.nan

# --- Realized Volatility ---

def realized_volatility(prices, window=22):
    log_returns = np.log(prices / prices.shift(1))
    rv = log_returns.rolling(window).std() * np.sqrt(252)
    return rv

# --- Backtest Strategy (with rolling med/std) ---

def backtest_strategy(df, rv_window=22, stat_window=252):
    df['rv'] = realized_volatility(df['S'], rv_window)
    df['iv'] = df.apply(lambda row: implied_volatility(
        row['option_price'], row['S'], row['K'], row['T'], option_type=row['type']), axis=1)

    df['med_vol'] = df['rv'].rolling(stat_window).median()
    df['std_vol'] = df['rv'].rolling(stat_window).std()

    position = None
    trades = []
    entry_price = 0

    for i in range(len(df)):
        row = df.iloc[i]
        date = row.name
        iv = row['iv']
        med = row['med_vol']
        std = row['std_vol']

        if pd.isna(iv) or pd.isna(med) or pd.isna(std):
            continue

        long_th = med - 1.5 * std
        short_th = med + 1.5 * std
        close_th = 0.5 * std

        if position is None:
            if iv < long_th:
                position = 'long_vol'
                entry_price = row['option_price']
                trades.append((date, 'long_vol', row['option_price']))
            elif iv > short_th:
                position = 'short_vol'
                entry_price = row['option_price']
                trades.append((date, 'short_vol', row['option_price']))
        elif abs(iv - med) < close_th:
            pnl = row['option_price'] - entry_price if position == 'long_vol' else entry_price - row['option_price']
            trades.append((date, 'close', row['option_price'], pnl))
            position = None

    return pd.DataFrame(trades, columns=['date', 'action', 'option_price'] + ['pnl']*(len(trades[0])==4)), df

# --- Giả lập dữ liệu (mock) ---

def mock_data():
    np.random.seed(42)
    dates = pd.date_range("2020-01-01", "2023-12-31", freq='B')
    S = pd.Series(40000 + np.cumsum(np.random.normal(0, 300, len(dates))), index=dates)
    K = S.copy()
    T = pd.Series(30/365, index=dates)
    type_ = pd.Series("call", index=dates)

    sigma_real = 0.6
    option_prices = [
        bs_price(S[i], K[i], T[i], 0, sigma_real + np.random.normal(0, 0.05), 'call')
        for i in range(len(S))
    ]

    df = pd.DataFrame({'S': S, 'K': K, 'T': T, 'type': type_, 'option_price': option_prices})
    return df

# --- Run Everything ---

def run():
    df = mock_data()
    trades, df_full = backtest_strategy(df)

    print(trades)

    if 'pnl' in trades.columns:
        pnl_df = trades.dropna(subset=['pnl']).set_index('date')
        pnl_df['cum_pnl'] = pnl_df['pnl'].cumsum()
        pnl_df['cum_pnl'].plot(title="Cumulative PnL (Rolling Threshold)", figsize=(10, 4))
        plt.grid()
        plt.show()

run()


/tmp/ipython-input-6-414147545.py:88: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  bs_price(S[i], K[i], T[i], 0, sigma_real + np.random.normal(0, 0.05), 'call')
/tmp/ipython-input-6-414147545.py:88: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  bs_price(S[i], K[i], T[i], 0, sigma_real + np.random.normal(0, 0.05), 'call')


        date     action  option_price
0 2021-01-18  short_vol   2429.403421
